# Cross-Entropy Baseline Analysis
## Comparing CE-only baseline vs Dice+CE baseline as requested by reviewers

In [14]:
import os
import pandas as pd
import numpy as np
from tabulate import tabulate
import warnings
from scipy.stats import ttest_rel

In [15]:
# Per case metrics (raw):
TEST_METRICS_CSVS = {
    "micro_ACE": "micro_ace_raw.csv",
    "macro_ACE": "macro_ace_raw.csv",
    "micro_ECE": "micro_ece_raw.csv",
    "macro_ECE": "macro_ece_raw.csv",
    "micro_MCE": "micro_mce_raw.csv",
    "macro_MCE": "macro_mce_raw.csv",
    "DSC": "mean_dice_raw.csv",
}

# Metrics summary:
TEST_METRICS_CSVS_SUMMARY = {
    "micro_ACE": "micro_ace_summary.csv",
    "macro_ACE": "macro_ace_summary.csv",
    "micro_ECE": "micro_ece_summary.csv",
    "macro_ECE": "macro_ece_summary.csv",
    "micro_MCE": "micro_mce_summary.csv",
    "macro_MCE": "macro_mce_summary.csv",
    "DSC": "mean_dice_summary.csv",
}

SEED = 12345

In [16]:
# CE-only baseline runs
RUNS_CE = {
    "acdc": "../bundles/acdc17_baseline_ce_2",
    "amos": "../bundles/amos22_baseline_ce_nl",
    "kits": "../bundles/kits23_baseline_ce_nl",
    "brats": "../bundles/brats21_baseline_ce_nl",
}

# Dice+CE baseline runs for comparison
RUNS_DICE_CE = {
    "acdc": "../bundles/acdc17_baseline_dice_ce_2",
    "amos": "../bundles/amos22_baseline_dice_ce_nl",
    "kits": "../bundles/kits23_baseline_dice_ce_nl",
    "brats": "../bundles/brats21_baseline_dice_ce_nl",
}

In [17]:
def load_csv_files(run_path):
    """Load CSV files for a given run path."""
    dataframes = {}
    
    # Load raw CSV files
    for metric, csv_file in TEST_METRICS_CSVS.items():
        csv_path = os.path.join(run_path, f'seed_{SEED}', "inference_results", csv_file)
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            if metric not in dataframes:
                dataframes[metric] = {}
            dataframes[metric]["raw"] = df
        else:
            warnings.warn(f"File does not exist: {csv_path}")
    
    # Load summary CSV files
    for metric, csv_file in TEST_METRICS_CSVS_SUMMARY.items():
        csv_path = os.path.join(run_path, f'seed_{SEED}', "inference_results", csv_file)
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            if metric not in dataframes:
                dataframes[metric] = {}
            dataframes[metric]["summary"] = df
        else:
            warnings.warn(f"File does not exist: {csv_path}")
    
    return dataframes


def load_baseline_results(runs_dict):
    """Load all baseline results."""
    results = {}
    for dataset, run_path in runs_dict.items():
        results[dataset] = load_csv_files(run_path)
        print(f"Loaded {dataset}")
    return results

In [18]:
def create_baseline_comparison_table(ce_results, avg_type="micro"):
    """Create a table showing CE baseline results for all datasets."""
    def get_metrics(dataframes, dataset_key, metric):
        try:
            df = dataframes[dataset_key][metric]["summary"]
            mean = df[df["class"] == "mean"]["mean"].values[0]
            std = df[df["class"] == "mean"]["std"].values[0]
            return mean, std
        except (KeyError, IndexError) as e:
            print(f"Error getting {metric} for {dataset_key}: {e}")
            return None, None

    table_data = []
    
    dataset_names = {
        "acdc": "ACDC 17",
        "amos": "AMOS 22",
        "kits": "KiTS 23",
        "brats": "BraTS 21"
    }
    
    for dataset in ["acdc", "amos", "kits", "brats"]:
        row = [dataset_names[dataset]]
        
        # DSC
        dsc_mean, dsc_std = get_metrics(ce_results, dataset, "DSC")
        if dsc_mean is not None:
            row.append(f"{dsc_mean:.3f} ± {dsc_std:.3f}")
        else:
            row.append("N/A")
        
        # ACE
        ace_mean, ace_std = get_metrics(ce_results, dataset, f"{avg_type}_ACE")
        if ace_mean is not None:
            row.append(f"{ace_mean:.3f} ± {ace_std:.3f}")
        else:
            row.append("N/A")
        
        # ECE
        ece_mean, ece_std = get_metrics(ce_results, dataset, f"{avg_type}_ECE")
        if ece_mean is not None:
            row.append(f"{ece_mean:.3f} ± {ece_std:.3f}")
        else:
            row.append("N/A")
        
        # MCE
        mce_mean, mce_std = get_metrics(ce_results, dataset, f"{avg_type}_MCE")
        if mce_mean is not None:
            row.append(f"{mce_mean:.3f} ± {mce_std:.3f}")
        else:
            row.append("N/A")
        
        table_data.append(row)

    headers = [
        "Dataset",
        "DSC",
        f"{avg_type.upper()} ACE",
        f"{avg_type.upper()} ECE",
        f"{avg_type.upper()} MCE",
    ]
    
    print(f"\n=== CE Baseline Results ({avg_type.upper()}) ===")
    print(tabulate(table_data, headers=headers, tablefmt="pipe"))

In [19]:
def calculate_baseline_p_values(ce_results, dice_ce_results):
    """Calculate p-values comparing CE baseline to Dice+CE baseline."""
    metrics_to_test = ["DSC", "macro_ACE", "macro_ECE", "macro_MCE"]
    datasets = ["acdc", "amos", "kits", "brats"]
    
    print("\n=== Statistical Significance Tests: CE vs Dice+CE Baseline (Paired t-test) ===\n")
    
    for dataset in datasets:
        print(f"\n--- {dataset.upper()} ---")
        results_table = []
        
        for metric in metrics_to_test:
            try:
                # Check if data exists
                if metric not in ce_results[dataset] or "raw" not in ce_results[dataset][metric]:
                    results_table.append([metric, "N/A"])
                    continue
                
                if metric not in dice_ce_results[dataset] or "raw" not in dice_ce_results[dataset][metric]:
                    results_table.append([metric, "N/A"])
                    continue
                
                # Load raw data (per-case values)
                df_ce = ce_results[dataset][metric]["raw"]
                df_dice_ce = dice_ce_results[dataset][metric]["raw"]
                
                # Extract mean column values
                ce_vals = df_ce["mean"].values
                dice_ce_vals = df_dice_ce["mean"].values
                
                # Perform paired t-test
                try:
                    _, p_val = ttest_rel(dice_ce_vals, ce_vals)
                except Exception as e:
                    print(f"  Error in t-test for {metric}: {e}")
                    p_val = np.nan
                
                # Format p-value
                if not np.isnan(p_val):
                    p_str = f"{p_val:.3e}"
                    # Add significance markers
                    if p_val < 0.001:
                        p_str += " ***"
                    elif p_val < 0.01:
                        p_str += " **"
                    elif p_val < 0.05:
                        p_str += " *"
                else:
                    p_str = "N/A"
                
                results_table.append([metric, p_str])
                
            except Exception as e:
                print(f"  Error processing {metric}: {e}")
                results_table.append([metric, "Error"])
        
        headers = ["Metric", "Dice+CE vs CE p-value"]
        print(tabulate(results_table, headers=headers, tablefmt="pipe"))
    
    print("\nSignificance levels: * p<0.05, ** p<0.01, *** p<0.001")

## Load Data

In [20]:
ce_results = load_baseline_results(RUNS_CE)
dice_ce_results = load_baseline_results(RUNS_DICE_CE)

Loaded acdc
Loaded amos
Loaded kits
Loaded brats
Loaded acdc
Loaded amos
Loaded kits
Loaded brats


## Macro Results

In [21]:
create_baseline_comparison_table(ce_results, avg_type="macro")


=== CE Baseline Results (MACRO) ===
| Dataset   | DSC           | MACRO ACE     | MACRO ECE     | MACRO MCE     |
|:----------|:--------------|:--------------|:--------------|:--------------|
| ACDC 17   | 0.868 ± 0.039 | 0.130 ± 0.028 | 0.002 ± 0.001 | 0.282 ± 0.061 |
| AMOS 22   | 0.872 ± 0.045 | 0.103 ± 0.024 | 0.000 ± 0.000 | 0.225 ± 0.049 |
| KiTS 23   | 0.853 ± 0.139 | 0.160 ± 0.080 | 0.002 ± 0.007 | 0.304 ± 0.143 |
| BraTS 21  | 0.895 ± 0.130 | 0.142 ± 0.063 | 0.001 ± 0.001 | 0.273 ± 0.111 |


## Micro Results

In [22]:
create_baseline_comparison_table(ce_results, avg_type="micro")


=== CE Baseline Results (MICRO) ===
| Dataset   | DSC           | MICRO ACE     | MICRO ECE     | MICRO MCE     |
|:----------|:--------------|:--------------|:--------------|:--------------|
| ACDC 17   | 0.868 ± 0.039 | 0.109 ± 0.000 | 0.002 ± 0.000 | 0.206 ± 0.000 |
| AMOS 22   | 0.872 ± 0.045 | 0.053 ± 0.000 | 0.000 ± 0.000 | 0.102 ± 0.000 |
| KiTS 23   | 0.853 ± 0.139 | 0.139 ± 0.000 | 0.002 ± 0.000 | 0.318 ± 0.000 |
| BraTS 21  | 0.895 ± 0.130 | 0.047 ± 0.000 | 0.000 ± 0.000 | 0.086 ± 0.000 |


## Statistical Significance: CE vs Dice+CE Baseline

In [23]:
calculate_baseline_p_values(ce_results, dice_ce_results)


=== Statistical Significance Tests: CE vs Dice+CE Baseline (Paired t-test) ===


--- ACDC ---
| Metric    | Dice+CE vs CE p-value   |
|:----------|:------------------------|
| DSC       | 3.783e-02 *             |
| macro_ACE | 3.564e-03 **            |
| macro_ECE | 3.017e-02 *             |
| macro_MCE | 5.132e-04 ***           |

--- AMOS ---
| Metric    | Dice+CE vs CE p-value   |
|:----------|:------------------------|
| DSC       | 1.177e-12 ***           |
| macro_ACE | 9.749e-03 **            |
| macro_ECE | 2.454e-03 **            |
| macro_MCE | 1.808e-06 ***           |

--- KITS ---
| Metric    |   Dice+CE vs CE p-value |
|:----------|------------------------:|
| DSC       |                  0.5512 |
| macro_ACE |                  0.574  |
| macro_ECE |                  0.2975 |
| macro_MCE |                  0.2098 |

--- BRATS ---
| Metric    | Dice+CE vs CE p-value   |
|:----------|:------------------------|
| DSC       | 7.487e-04 ***           |
| macro_ACE | 3.904e-0